# Scoring fan sentiment

Turns the raw comment archive into one number per pick: how this fanbase
reacted to this selection, relative to how it reacts to selections in general.

**Why VADER.** The corpus is short, slangy, emoji-laden social-media text —
the register VADER was built for — and it is deterministic and fast enough to
score several hundred thousand comments in minutes. It is not subtle: draft
euphoria often arrives as profanity ("GET THE FUCK IN HERE" scores *negative*)
and sarcasm goes undetected. Those errors are noise at the comment level; the
unit of analysis here is the pick, averaging hundreds of comments, and the
final signal is standardized within fanbase, which absorbs each community's
baseline of profane enthusiasm.

**Cleaning.** Deleted and removed bodies carry no text, and AutoModerator
posts carry no fan opinion; both are dropped. Everything else is kept —
draft-night comments are short by nature and length filters would discard
real reactions ("YES", "ugh").

In [1]:
import json
from pathlib import Path

import pandas as pd
from nltk.sentiment.vader import SentimentIntensityAnalyzer

COMMENTS_DIR = Path("..") / "data" / "raw" / "reddit_comments"
PROCESSED = Path("..") / "data" / "processed"

matched = pd.read_csv(PROCESSED / "matched_threads.csv")
pick_of = matched.set_index("post_id")[
    ["season", "team", "round", "pick", "pfr_player_name"]]

rows = []
for path in COMMENTS_DIR.glob("*.json"):
    if path.stem not in pick_of.index:
        continue
    meta = pick_of.loc[path.stem]
    for c in json.loads(path.read_text()):
        rows.append((meta.season, meta.team, meta["round"], meta.pick,
                     meta.pfr_player_name, path.stem,
                     c.get("author", ""), c.get("body", "") or ""))

comments = pd.DataFrame(rows, columns=[
    "season", "team", "round", "pick", "pfr_player_name",
    "post_id", "author", "body"])
print(f"comments loaded: {len(comments):,} across "
      f"{comments.post_id.nunique():,} threads")

dead = comments.body.isin(["[deleted]", "[removed]", ""])
bots = comments.author.isin(["AutoModerator"])
comments = comments[~dead & ~bots]
print(f"after cleaning: {len(comments):,} "
      f"({dead.sum():,} deleted/removed/empty, {bots.sum():,} bot)")

comments loaded: 402,999 across 5,485 threads
after cleaning: 388,026 (14,687 deleted/removed/empty, 286 bot)


In [2]:
sia = SentimentIntensityAnalyzer()
comments["compound"] = [sia.polarity_scores(b)["compound"]
                        for b in comments.body]
print(comments["compound"].describe().round(3).to_string())

count    388026.000
mean          0.189
std           0.451
min          -0.996
25%           0.000
50%           0.083
75%           0.572
max           1.000


### From comments to a per-pick signal

**Aggregation: one commenter, one vote.** Each pick's sentiment is the plain
mean of its comments' compound scores, pooled across that pick's threads.
Upvote-weighting was considered and rejected: archived scores are a snapshot
taken well after draft night, so weighting by them would leak post-draft
hindsight into what must remain a draft-night measure.

**Standardization: within fanbase and class.** Fanbases love their own picks
— nearly every mean is positive — and baseline enthusiasm differs by
community and by year. Each pick's sentiment is therefore z-scored against
the other picks *the same fanbase made in the same class*, making the signal
"which of its own picks was this fanbase unusually high or low on." The cost
is honest: with 7–11 picks per team-year, these z-scores are estimated from
small samples, which the modeling stage must respect.

In [3]:
pick_sent = (comments.groupby(
    ["season", "team", "round", "pick", "pfr_player_name"])
    .agg(n_comments=("compound", "size"),
         sent_mean=("compound", "mean"))
    .reset_index())

grp = pick_sent.groupby(["season", "team"])["sent_mean"]
pick_sent["sent_z"] = (pick_sent["sent_mean"] - grp.transform("mean")) \
    / grp.transform("std")

print(f"picks with sentiment: {len(pick_sent)}")
print("\nshare of picks with positive mean sentiment:",
      f"{(pick_sent.sent_mean > 0).mean():.1%}")
print("\ncomments per pick:")
print(pick_sent["n_comments"].describe().round(1).to_string())
print("\nmean sentiment by round (fan optimism gradient):")
print(pick_sent.groupby("round")["sent_mean"].mean().round(3).to_string())

picks with sentiment: 1531

share of picks with positive mean sentiment: 99.3%

comments per pick:
count    1531.0
mean      253.4
std       304.8
min         1.0
25%        62.0
50%       141.0
75%       323.0
max      2176.0

mean sentiment by round (fan optimism gradient):
round
1    0.191
2    0.202
3    0.208
4    0.219
5    0.227
6    0.209
7    0.222


In [4]:
# Face validity: the class's extremes should read like stories fans remember
for season in sorted(pick_sent.season.unique()):
    sub = pick_sent[(pick_sent.season == season) & (pick_sent.n_comments >= 30)]
    hi = sub.nlargest(3, "sent_z")
    lo = sub.nsmallest(3, "sent_z")
    print(f"=== {season} ===")
    for _, r in pd.concat([hi, lo]).iterrows():
        print(f"  {r.sent_z:+.2f}  {r.team} #{r['pick']:<3.0f} "
              f"{r.pfr_player_name}  ({r.n_comments} comments)")

=== 2021 ===
  +2.50  JAX #106 Jay Tufele  (87 comments)
  +2.10  IND #165 Shawn Davis  (41 comments)
  +2.08  NYJ #186 Hamsah Nasirildeen  (100 comments)
  -2.09  DET #72  Alim McNeill  (51 comments)
  -2.08  DAL #138 Josh Ball  (149 comments)
  -1.98  CIN #122 Tyler Shelvin  (112 comments)
=== 2022 ===
  +1.91  SFO #220 Kalia Davis  (86 comments)
  +1.83  LAR #142 Cobie Durant  (50 comments)
  +1.77  IND #216 Curtis Brooks  (44 comments)
  -2.06  BAL #139 Isaiah Likely  (115 comments)
  -2.05  DET #217 James Houston  (129 comments)
  -2.01  CHI #207 Doug Kramer  (115 comments)
=== 2023 ===
  +2.07  SEA #154 Olusegun Oluwatimi  (171 comments)
  +1.93  LVR #203 Amari Burney  (103 comments)
  +1.88  PIT #132 Nick Herbig  (159 comments)
  -1.96  CIN #60  DJ Turner  (251 comments)
  -1.86  DET #152 Colby Sorsdal  (185 comments)
  -1.73  BUF #252 Alex Austin  (37 comments)
=== 2024 ===
  +2.38  LVR #223 Trey Taylor  (62 comments)
  +2.34  DAL #216 Ryan Flournoy  (66 comments)
  +2.16  NWE 

In [5]:
out_path = PROCESSED / "pick_sentiment.csv"
pick_sent.to_csv(out_path, index=False)
print(f"wrote {len(pick_sent)} rows to {out_path}")

wrote 1531 rows to ../data/processed/pick_sentiment.csv
